# Smart Turn v3.2 — JupyterHub GPU benchmark

This notebook benchmarks the official unquantised GPU model on labelled prerecorded audio. It measures Smart Turn only: FastAPI, Whisper, Gemma, retrieval, and the main LLM are not started.

In [ ]:
import platform
import socket
import sys

print("Host:", socket.gethostname())
print("Python:", sys.version)
print("Platform:", platform.platform())

## Install inference dependencies

Run this before importing ONNX Runtime. If the kernel previously imported a CPU-only `onnxruntime`, restart the kernel after this cell and continue from the next cell.

In [ ]:
%pip install -q --upgrade onnxruntime-gpu numpy librosa soundfile transformers huggingface_hub

In [ ]:
import onnxruntime as ort

providers = ort.get_available_providers()
print("ONNX Runtime:", ort.__version__)
print("Available providers:", providers)

if "CUDAExecutionProvider" not in providers:
    raise RuntimeError(
        "CUDAExecutionProvider is unavailable. Confirm that this notebook "
        "is attached to the GPU kernel, then restart the kernel after the "
        "onnxruntime-gpu installation cell."
    )

## Run the labelled benchmark

Five warm-up predictions are discarded. The following 30 runs are measured, giving a stable median and p95 for ONNX inference and the complete local prediction path.

In [ ]:
%run benchmark_smart_turn.py --model gpu --warmup-runs 5 --runs 30 --output results/smart_turn_gpu_results.json

In [ ]:
import json
from pathlib import Path

result_path = Path("results/smart_turn_gpu_results.json")
results = json.loads(result_path.read_text(encoding="utf-8"))

for case in results["cases"]:
    print(
        case["case_id"],
        {
            "expected": case["expected_label"],
            "decision": case["decision"],
            "probability_complete": case["probability_complete"],
            "inference": case["timings"]["inference_ms"],
            "end_to_end": case["timings"]["end_to_end_ms"],
        },
    )